## Implementing Recurrent Neural Networks in PyTorch


Recurrent Neural Networks (RNNs) are neural networks that are particularly effective for sequential data. Unlike traditional feedforward neural networks RNNs have connections that form loops allowing them to maintain a hidden state that can capture information from previous inputs. This makes them suitable for tasks such as time series prediction, natural language processing and many more task.

In this notebook we will implement a simple RNN using PyTorch to perform a binary classification task on a text dataset. 

Before we start implementing the RNN we need to set up our environment. Ensure you have PyTorch installed. You can install it using pip:

!pip install torch


### Classifying Movie Reviews Using RNN

### 1. Importing Libraries

In [2]:
# %pip install torch
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader

### 2. Loading and Preprocessing the Dataset

In [7]:
df = pd.read_csv("IMDB-Dataset.csv")

df.head(3)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive


In [8]:
df['review'] = df['review'].str.lower().str.split()

le = LabelEncoder()

df['sentiment'] = le.fit_transform(df['sentiment'])



In [11]:
df.shape

(50000, 2)

In [9]:
df['review'].head(3)

0    [one, of, the, other, reviewers, has, mentione...
1    [a, wonderful, little, production., <br, /><br...
2    [i, thought, this, was, a, wonderful, way, to,...
Name: review, dtype: object

In [10]:
df['sentiment'].head(3)

0    1
1    1
2    1
Name: sentiment, dtype: int64

In [12]:
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42 )

vocab = {word for phrase in df['review'] for word in phrase}
word_to_idx = {word: idx for idx, word in enumerate(vocab, start=1)}

max_length = df['review'].str.len().max()

def encode_and_pad(text):
    encoded = [word_to_idx.get(word, 0) for word in text]
    padded = encoded + [0] * (max_length - len(encoded))
    return padded

train_data['review'] = train_data['review'].apply(encode_and_pad)
test_data['review'] = test_data['review'].apply(encode_and_pad)


### 3. Creating Dataset and Data Loader

* Define a custom SentimentDataset class inheriting from PyTorch’s Dataset.
* Store texts and labels from input data within the class.
* Implement __len__ method to return total number of samples.
* Implement __getitem__ method to retrieve a single sample by index, converting text and label to PyTorch tensors with correct data types.
* Create dataset instances for training and testing data.
* Wrap datasets in DataLoaders with a batch size of 32.
* Shuffle training data in DataLoader for randomness, keep test data ordered.
* Prepare data for efficient batch loading during model training and evaluation.

In [13]:
class SentimentDataset(Dataset):
    def __init__(self, data):
        self.texts = data['review'].values
        self.labels = data['sentiment'].values

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        return torch.tensor(text, dtype=torch.long), torch.tensor(label, dtype=torch.float)
    
train_dataset = SentimentDataset(train_data)
test_dataset = SentimentDataset(test_data)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### 4. Defining the RNN Model

In [14]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, output_size):
        super(SentimentRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.embedding(x)
        h0 = torch.zeros(1, x.size(0), hidden_size).to(x.device)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out
    
vocab_size = len(vocab) + 1
embed_size = 128
hidden_size = 64
output_size = 2
model = SentimentRNN(vocab_size, embed_size, hidden_size, output_size)


### 5. Training the Model

In [17]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for texts, labels in train_loader:
        outputs = model(texts)
        loss = criterion(outputs, labels.long())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss/len(train_loader):.4f}')

Epoch 1/10, Loss: 0.6957
Epoch 2/10, Loss: 0.6942


KeyboardInterrupt: 

### 6. Evaluating the Model

In [ ]:
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for texts, labels in test_loader:
        outputs = model(texts)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy: {100 * correct / total:.2f}%')